# LeetCode #74 — Search a 2D Matrix: binary search trên ma trận "trải phẳng"

**Mức độ:** Medium · **Chủ đề:** Array, Binary Search, Matrix

Skeleton trên LeetCode:

```python
class Solution:
    def searchMatrix(self, matrix: List[List[int]], target: int) -> bool:
```

Mục tiêu của notebook:

1. Nhận ra ma trận này thực chất là **một mảng đã sắp xếp** bị cắt thành từng hàng.
2. Chuyển đổi chỉ số **1 chiều ↔ 2 chiều** bằng `divmod`.
3. Viết **binary search** chuẩn, không lỗi biên.
4. So sánh với hai cách khác: hai lần binary search, và "đi cầu thang" từ góc.

# 1. Bài toán là gì?

Cho ma trận số nguyên `m × n` có hai tính chất:

- Mỗi hàng được sắp xếp **không giảm**.
- Phần tử đầu mỗi hàng **lớn hơn** phần tử cuối của hàng trước.

Cho `target`, trả về `True` nếu `target` có trong ma trận. Yêu cầu: **`O(log(m · n))`** thời gian.

**Constraints (theo đề):**

- `m == matrix.length`, `n == matrix[i].length`
- `1 <= m, n <= 100`
- `-10^4 <= matrix[i][j], target <= 10^4`

---

## Ví dụ

```text
matrix = [[1, 3, 5, 7],
          [10, 11, 16, 20],
          [23, 30, 34, 60]]

target = 3   -> True
target = 13  -> False
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Tìm kiếm nhị phân (binary search)
2. Ma trận (mảng hai chiều) trong Python

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Tìm kiếm nhị phân (binary search)

### Ý tưởng

Trên mảng **đã sắp xếp**, so phần tử cần tìm với phần tử **ở giữa**: nếu nhỏ hơn thì chắc chắn nằm ở nửa trái, lớn hơn thì ở nửa phải. Mỗi bước loại **một nửa** → chỉ `O(log n)` bước (mảng 1 triệu phần tử: ~20 bước).

```text
tìm 23 trong [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
lo=0 hi=9 mid=4 → 16 < 23 → lo = 5
lo=5 hi=9 mid=7 → 56 > 23 → hi = 6
lo=5 hi=6 mid=5 → 23 = 23 → tìm thấy ở chỉ số 5
```

In [1]:
def tim_nhi_phan(a, x):
    lo, hi = 0, len(a) - 1                # khoảng tìm đóng [lo, hi]
    while lo <= hi:
        mid = (lo + hi) // 2
        print(f"   lo={lo} hi={hi} mid={mid} a[mid]={a[mid]}")
        if a[mid] == x:
            return mid
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

a_bs = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print("→", tim_nhi_phan(a_bs, 23))
print("→", tim_nhi_phan(a_bs, 7), "(không có)")

   lo=0 hi=9 mid=4 a[mid]=16
   lo=5 hi=9 mid=7 a[mid]=56
   lo=5 hi=6 mid=5 a[mid]=23
→ 5
   lo=0 hi=9 mid=4 a[mid]=16
   lo=0 hi=3 mid=1 a[mid]=5
   lo=2 hi=3 mid=2 a[mid]=8
→ -1 (không có)


### Dạng tổng quát: tìm vị trí **đầu tiên** thỏa điều kiện

Rất nhiều bài không hỏi "có `x` không" mà hỏi "vị trí đầu tiên mà điều kiện `P(i)` đúng", với `P` **đơn điệu**: `False False … False True True … True`. Khung:

```text
lo, hi = 0, n            # đáp án nằm trong [lo, hi]; n nghĩa là "không có"
while lo < hi:
    mid = (lo + hi) // 2
    if P(mid): hi = mid          # mid có thể là đáp án → giữ lại
    else:      lo = mid + 1      # mid chắc chắn không phải
return lo
```

Hai trường hợp quan trọng:

- **lower_bound**: vị trí đầu tiên `a[i] >= x` (`bisect_left`).
- **upper_bound**: vị trí đầu tiên `a[i] > x` (`bisect_right`).

Số lần xuất hiện của `x` = `upper_bound − lower_bound`.

In [2]:
from bisect import bisect_left, bisect_right

def dau_tien(n, P):
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi) // 2
        if P(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

b_bs = [1, 2, 2, 2, 5, 7]
lb = dau_tien(len(b_bs), lambda i: b_bs[i] >= 2)
ub = dau_tien(len(b_bs), lambda i: b_bs[i] > 2)
print("lower_bound(2) =", lb, "| upper_bound(2) =", ub, "| số lần xuất hiện:", ub - lb)
print("bisect_left / bisect_right:", bisect_left(b_bs, 2), bisect_right(b_bs, 2))
print("chèn 6 vào vị trí:", bisect_left(b_bs, 6))
print("số nguyên nhỏ nhất k với k*k >= 50:", dau_tien(100, lambda k: k * k >= 50))

lower_bound(2) = 1 | upper_bound(2) = 4 | số lần xuất hiện: 3
bisect_left / bisect_right: 1 4
chèn 6 vào vị trí: 5
số nguyên nhỏ nhất k với k*k >= 50: 8


### Lỗi hay gặp (nguồn lỗi số 1 là "lệch 1")

- Trộn hai kiểu khoảng: `while lo <= hi` đi với `hi = mid - 1`; `while lo < hi` đi với `hi = mid`. Trộn lẫn → **lặp vô hạn** hoặc bỏ sót.
- `lo = mid` (thay vì `mid + 1`) với `mid = (lo+hi)//2` → khi `hi = lo + 1` thì `mid = lo`, không tiến → lặp vô hạn.
- Áp dụng trên mảng **chưa sắp xếp** / điều kiện không đơn điệu.
- (Ngôn ngữ khác) `(lo + hi) / 2` tràn số → dùng `lo + (hi - lo) // 2`; Python không bị.

In [3]:
def sai_lo_bang_mid(a, x, gioi_han=20):
    lo, hi, buoc = 0, len(a) - 1, 0
    while lo < hi and buoc < gioi_han:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid            # SAI: không loại mid
        else:
            hi = mid
        buoc += 1
    return buoc
print("lo = mid: số vòng trước khi bị chặn:", sai_lo_bang_mid([1, 3], 3))

lo = mid: số vòng trước khi bị chặn: 20


## 2.2 Ma trận (mảng hai chiều) trong Python

### Tạo và truy cập

Ma trận `m × n` là list gồm `m` hàng, mỗi hàng là list `n` phần tử. Ô hàng `i`, cột `j` là `M[i][j]`; số hàng `len(M)`, số cột `len(M[0])`.

```text
        j=0 j=1 j=2
i=0   [  1,  2,  3 ]
i=1   [  4,  5,  6 ]
```

In [4]:
M_mx = [[1, 2, 3],
        [4, 5, 6]]
print("M[1][2] =", M_mx[1][2], "| số hàng:", len(M_mx), "| số cột:", len(M_mx[0]))

dung_mx = [[0] * 3 for _ in range(2)]    # ĐÚNG: mỗi hàng là list riêng
sai_mx = [[0] * 3] * 2                   # SAI: 2 hàng là cùng một list
dung_mx[0][0] = sai_mx[0][0] = 7
print("tạo bằng for:", dung_mx, "| nhân list:", sai_mx)

M[1][2] = 6 | số hàng: 2 | số cột: 3
tạo bằng for: [[7, 0, 0], [0, 0, 0]] | nhân list: [[7, 0, 0], [7, 0, 0]]


### Duyệt và các phép biến đổi cơ bản

- Duyệt theo hàng: `for i in range(m): for j in range(n)`.
- **Chuyển vị** (transpose): `T[j][i] = M[i][j]` — hàng thành cột. Trong Python: `list(zip(*M))`.
- **Lật ngang** một hàng: `row[::-1]`; lật dọc: đảo thứ tự hàng `M[::-1]`.
- Xoay 90° theo chiều kim đồng hồ = chuyển vị rồi lật từng hàng.

In [5]:
def in_mt(A):
    for row in A:
        print("   ", " ".join(f"{v:>2}" for v in row))

A_mx = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("gốc:"); in_mt(A_mx)
T_mx = [list(r) for r in zip(*A_mx)]
print("chuyển vị:"); in_mt(T_mx)
print("xoay 90° phải (chuyển vị + lật hàng):"); in_mt([r[::-1] for r in T_mx])

gốc:
     1  2  3
     4  5  6
     7  8  9
chuyển vị:
     1  4  7
     2  5  8
     3  6  9
xoay 90° phải (chuyển vị + lật hàng):
     7  4  1
     8  5  2
     9  6  3


### Ô kề và mảng hướng

Bốn ô kề của `(r, c)`: lên `(r−1, c)`, xuống `(r+1, c)`, trái `(r, c−1)`, phải `(r, c+1)`. Viết thành **mảng hướng** để khỏi lặp code, và luôn **kiểm tra biên** `0 <= nr < m and 0 <= nc < n`.

In [6]:
HUONG = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # lên, xuống, trái, phải
def o_ke(r, c, m, n):
    return [(r + dr, c + dc) for dr, dc in HUONG if 0 <= r + dr < m and 0 <= c + dc < n]
print("ô kề của (0, 0) trong lưới 3×3:", o_ke(0, 0, 3, 3))
print("ô kề của (1, 1):", o_ke(1, 1, 3, 3))

ô kề của (0, 0) trong lưới 3×3: [(1, 0), (0, 1)]
ô kề của (1, 1): [(0, 1), (2, 1), (1, 0), (1, 2)]


### Đánh số ô thành một chỉ số và ngược lại

Ma trận `m × n` "trải phẳng" theo hàng: ô `(i, j)` ↔ chỉ số `k = i * n + j`; ngược lại `i, j = divmod(k, n)`. Hữu ích để tìm nhị phân trên ma trận hoặc lưu ô vào set.

In [7]:
n_cols = 4
for k in [0, 5, 11]:
    print(f"k = {k:>2} → (i, j) = {divmod(k, n_cols)} → k = {divmod(k, n_cols)[0] * n_cols + divmod(k, n_cols)[1]}")

k =  0 → (i, j) = (0, 0) → k = 0
k =  5 → (i, j) = (1, 1) → k = 5
k = 11 → (i, j) = (2, 3) → k = 11


Lỗi hay gặp: `[[0]*n]*m`; nhầm `m` (hàng) với `n` (cột) khi ma trận không vuông; quên kiểm tra biên; sửa ma trận trong lúc vẫn cần giá trị cũ để tính ô khác.

## 2.3 Áp dụng vào bài này

Ma trận đọc theo hàng là một dãy tăng dần → **trải phẳng** chỉ số `k ↔ (k // n, k % n)` rồi **tìm nhị phân** trên `0..m·n−1` như mảng thường.

# 3. Nền tảng

## 3.1 Ma trận = mảng sắp xếp bị cắt

Hai tính chất trên nghĩa là: đọc ma trận **từng hàng nối tiếp nhau**, ta được một mảng tăng dần.

In [8]:
from typing import List

matrix = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]]
flat = [x for row in matrix for x in row]
print(flat)
print("Đã sắp xếp?", flat == sorted(flat))

[1, 3, 5, 7, 10, 11, 16, 20, 23, 30, 34, 60]
Đã sắp xếp? True


## 3.2 Chỉ số 1 chiều ↔ 2 chiều

Với `n` cột, phần tử thứ `k` (đếm từ 0) của mảng phẳng nằm ở:

```text
hàng = k // n,  cột = k % n          (dùng divmod(k, n))
ngược lại: k = hàng * n + cột
```

In [9]:
n = 4
for k in [0, 3, 4, 6, 11]:
    r, c = divmod(k, n)
    print(f"k = {k:>2} -> (hàng {r}, cột {c}) -> giá trị {matrix[r][c]:>2} | kiểm tra flat[k] = {flat[k]}")

k =  0 -> (hàng 0, cột 0) -> giá trị  1 | kiểm tra flat[k] = 1
k =  3 -> (hàng 0, cột 3) -> giá trị  7 | kiểm tra flat[k] = 7
k =  4 -> (hàng 1, cột 0) -> giá trị 10 | kiểm tra flat[k] = 10
k =  6 -> (hàng 1, cột 2) -> giá trị 16 | kiểm tra flat[k] = 16
k = 11 -> (hàng 2, cột 3) -> giá trị 60 | kiểm tra flat[k] = 60


Như vậy ta **không cần** tạo mảng phẳng (tốn `O(m·n)`): truy cập phần tử thứ `k` trực tiếp bằng `matrix[k // n][k % n]`.

# 4. Baseline: duyệt hết

`O(m · n)` — đúng, nhưng không đạt yêu cầu `O(log(m·n))`.

In [10]:
def search_linear(matrix: List[List[int]], target: int) -> bool:
    return any(target in row for row in matrix)

print(search_linear(matrix, 3), search_linear(matrix, 13))

True False


# 5. Lời giải chính: binary search trên `[0, m·n − 1]`

```text
lo, hi = 0, m*n - 1
while lo <= hi:
    mid = (lo + hi) // 2
    x = matrix[mid // n][mid % n]
    if x == target: return True
    if x < target: lo = mid + 1     # target nằm bên phải
    else:          hi = mid - 1     # target nằm bên trái
return False
```

Bất biến: nếu `target` có trong ma trận thì nó nằm trong đoạn `[lo, hi]`.

In [11]:
def search_flat(matrix: List[List[int]], target: int) -> bool:
    m, n = len(matrix), len(matrix[0])
    lo, hi = 0, m * n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        r, c = divmod(mid, n)
        x = matrix[r][c]
        if x == target:
            return True
        if x < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

print(search_flat(matrix, 3), search_flat(matrix, 13))

True False


# 6. Mô phỏng

In [12]:
def trace_search(matrix: List[List[int]], target: int) -> bool:
    m, n = len(matrix), len(matrix[0])
    lo, hi = 0, m * n - 1
    print(f"target = {target}")
    while lo <= hi:
        mid = (lo + hi) // 2
        r, c = divmod(mid, n)
        x = matrix[r][c]
        if x == target:
            print(f"  lo={lo:>2} hi={hi:>2} mid={mid:>2} -> ({r},{c}) = {x:>2}  == target -> True")
            return True
        if x < target:
            print(f"  lo={lo:>2} hi={hi:>2} mid={mid:>2} -> ({r},{c}) = {x:>2}  < target -> lo = {mid + 1}")
            lo = mid + 1
        else:
            print(f"  lo={lo:>2} hi={hi:>2} mid={mid:>2} -> ({r},{c}) = {x:>2}  > target -> hi = {mid - 1}")
            hi = mid - 1
    print(f"  lo={lo} > hi={hi} -> False")
    return False

_ = trace_search(matrix, 3)
print()
_ = trace_search(matrix, 13)

target = 3
  lo= 0 hi=11 mid= 5 -> (1,1) = 11  > target -> hi = 4
  lo= 0 hi= 4 mid= 2 -> (0,2) =  5  > target -> hi = 1
  lo= 0 hi= 1 mid= 0 -> (0,0) =  1  < target -> lo = 1
  lo= 1 hi= 1 mid= 1 -> (0,1) =  3  == target -> True

target = 13
  lo= 0 hi=11 mid= 5 -> (1,1) = 11  < target -> lo = 6
  lo= 6 hi=11 mid= 8 -> (2,0) = 23  > target -> hi = 7
  lo= 6 hi= 7 mid= 6 -> (1,2) = 16  > target -> hi = 5
  lo=6 > hi=5 -> False


Với `m · n = 12` phần tử, mỗi lần tìm chỉ tốn tối đa `⌈log2(13)⌉ = 4` bước.

# 7. Hai cách khác

## 7.1 Hai lần binary search

Lần 1 tìm **hàng** có thể chứa `target` (hàng cuối có `row[0] <= target`); lần 2 tìm trong hàng đó. Cũng `O(log m + log n) = O(log(m·n))`. Dùng `bisect` của thư viện chuẩn:

In [13]:
import bisect

def search_two_bs(matrix: List[List[int]], target: int) -> bool:
    firsts = [row[0] for row in matrix]                  # O(m) để tạo list, chỉ minh họa
    r = bisect.bisect_right(firsts, target) - 1          # hàng cuối cùng có row[0] <= target
    if r < 0:
        return False
    row = matrix[r]
    c = bisect.bisect_left(row, target)
    return c < len(row) and row[c] == target

print(search_two_bs(matrix, 3), search_two_bs(matrix, 13), search_two_bs(matrix, 0))

True False False


(Tạo list `firsts` tốn `O(m)`; để đúng `O(log m)` thì tự viết binary search trên `matrix[r][0]`. Ở đây chỉ để minh họa `bisect`.)

## 7.2 "Đi cầu thang" từ góc trên phải — `O(m + n)`

Bắt đầu ở góc trên phải: nếu giá trị lớn hơn `target` → sang trái; nhỏ hơn → xuống dưới. Cách này **không** đạt `O(log)` nhưng chỉ cần mỗi hàng và mỗi cột tăng dần — dùng được cho **#240 Search a 2D Matrix II**, nơi cách "trải phẳng" **không** áp dụng được.

In [14]:
def search_staircase(matrix: List[List[int]], target: int) -> bool:
    r, c = 0, len(matrix[0]) - 1
    while r < len(matrix) and c >= 0:
        x = matrix[r][c]
        if x == target:
            return True
        if x > target:
            c -= 1
        else:
            r += 1
    return False

print(search_staircase(matrix, 3), search_staircase(matrix, 13))

True False


# 8. Phiên bản đúng format LeetCode

In [15]:
class Solution:
    def searchMatrix(self, matrix: List[List[int]], target: int) -> bool:
        m, n = len(matrix), len(matrix[0])
        lo, hi = 0, m * n - 1
        while lo <= hi:
            mid = (lo + hi) // 2
            x = matrix[mid // n][mid % n]
            if x == target:
                return True
            if x < target:
                lo = mid + 1
            else:
                hi = mid - 1
        return False

In [16]:
import random

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<24} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.searchMatrix(matrix, 3), True)
check("Ví dụ 2", sol.searchMatrix(matrix, 13), False)
check("1 x 1 có", sol.searchMatrix([[5]], 5), True)
check("1 x 1 không", sol.searchMatrix([[5]], 6), False)
check("Nhỏ hơn tất cả", sol.searchMatrix(matrix, -5), False)
check("Lớn hơn tất cả", sol.searchMatrix(matrix, 100), False)
check("Phần tử đầu", sol.searchMatrix(matrix, 1), True)
check("Phần tử cuối", sol.searchMatrix(matrix, 60), True)
check("Một cột", sol.searchMatrix([[1], [3], [5]], 3), True)

random.seed(74)
for _ in range(3000):
    m, n = random.randint(1, 6), random.randint(1, 6)
    values = sorted(random.sample(range(-50, 50), m * n))
    mat = [values[i * n:(i + 1) * n] for i in range(m)]
    t = random.randint(-55, 55)
    expected = search_linear(mat, t)
    assert sol.searchMatrix(mat, t) == expected == search_two_bs(mat, t) == search_staircase(mat, t)
print("3000 test ngẫu nhiên: 4 cách khớp nhau")

Ví dụ 1                  PASS  True
Ví dụ 2                  PASS  False
1 x 1 có                 PASS  True
1 x 1 không              PASS  False
Nhỏ hơn tất cả           PASS  False
Lớn hơn tất cả           PASS  False
Phần tử đầu              PASS  True
Phần tử cuối             PASS  True
Một cột                  PASS  True


3000 test ngẫu nhiên: 4 cách khớp nhau

(Đề cho phép hàng "không giảm" nên có thể có phần tử trùng trong một hàng; test trên dùng giá trị phân biệt cho gọn — binary search vẫn đúng khi có trùng lặp vì ta chỉ cần tìm **một** vị trí bằng `target`.)

# 9. Vì sao đúng? Độ phức tạp?

Mảng phẳng `flat[k] = matrix[k // n][k % n]` tăng dần (tính chất của đề). Binary search giữ bất biến "`target` (nếu có) nằm trong `[lo, hi]`":

- `flat[mid] < target` → mọi `k ≤ mid` đều `< target` → bỏ nửa trái: `lo = mid + 1`.
- `flat[mid] > target` → bỏ nửa phải: `hi = mid - 1`.

Mỗi vòng đoạn `[lo, hi]` giảm ít nhất một nửa → dừng sau `O(log(m·n))` vòng; khi `lo > hi` đoạn rỗng → không có `target`.

```text
Time:  O(log(m · n))
Space: O(1)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Dùng `m` thay vì `n` khi đổi chỉ số

Hàng = `k // n` (chia cho **số cột**), không phải `k // m`. Khi ma trận vuông thì không thấy lỗi!

In [17]:
def wrong_divide_by_m(matrix: List[List[int]], target: int) -> bool:
    m, n = len(matrix), len(matrix[0])
    lo, hi = 0, m * n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        r, c = divmod(mid, m)                     # SAI: phải là n
        if r >= m or c >= n:
            return False
        x = matrix[r][c]
        if x == target:
            return True
        if x < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

print("Sai :", [wrong_divide_by_m(matrix, t) for t in (1, 3, 11, 34, 60)])
print("Đúng:", [sol.searchMatrix(matrix, t) for t in (1, 3, 11, 34, 60)])

Sai : [True, True, True, True, False]
Đúng: [True, True, True, True, True]


## Lỗi 2: `while lo < hi` với `hi = mid - 1`

Trộn hai "trường phái" binary search sẽ bỏ sót phần tử khi `lo == hi`.

In [18]:
def wrong_loop_condition(matrix: List[List[int]], target: int) -> bool:
    m, n = len(matrix), len(matrix[0])
    lo, hi = 0, m * n - 1
    while lo < hi:                                # SAI: khi lo == hi chưa kiểm tra
        mid = (lo + hi) // 2
        x = matrix[mid // n][mid % n]
        if x == target:
            return True
        if x < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

print("Sai :", wrong_loop_condition([[5]], 5), "| Đúng:", sol.searchMatrix([[5]], 5))
print("Sai :", wrong_loop_condition(matrix, 60), "| Đúng:", sol.searchMatrix(matrix, 60))

Sai : False | Đúng: True
Sai : False | Đúng: True


## Lỗi 3: Chọn sai hàng ở cách "hai lần binary search"

Hàng cần tìm là hàng **cuối cùng** có `row[0] <= target` (dùng `bisect_right - 1`), không phải hàng đầu tiên có `row[0] >= target` — nếu `target` không phải phần tử đầu hàng, nó nằm ở hàng **trước**.

## Lỗi 4: Làm phẳng ma trận ra list mới

Đúng kết quả, nhưng tốn `O(m·n)` thời gian và bộ nhớ — mất hết lợi ích của binary search.

# 11. Mẹo nhận diện pattern

- Ma trận có thứ tự **"đọc theo hàng là tăng dần"** → trải phẳng ảo + binary search (`divmod`).
- Chỉ có **mỗi hàng tăng** và **mỗi cột tăng** (không nối tiếp) → đi cầu thang `O(m + n)` (#240).
- Tìm phần tử thứ `k` trong ma trận hàng/cột tăng → binary search trên **giá trị** + đếm (#378).

## Template binary search (đóng hai đầu)

```python
lo, hi = 0, N - 1
while lo <= hi:
    mid = (lo + hi) // 2
    if a(mid) == target: return True
    if a(mid) < target: lo = mid + 1
    else: hi = mid - 1
return False
```

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `matrix` ở ví dụ và `target = 30`, ghi lại `lo, hi, mid, (r, c)` từng bước.

## Bài 2 — Điền chỗ trống

In [19]:
def exercise_2_search_matrix(matrix: List[List[int]], target: int) -> bool:
    m, n = len(matrix), len(matrix[0])
    lo, hi = 0, m * n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        # TODO: đổi mid -> (r, c), so sánh và thu hẹp
        pass
    return False


# assert exercise_2_search_matrix([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 3) is True
# assert exercise_2_search_matrix([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 13) is False
# assert exercise_2_search_matrix([[1]], 1) is True

## Bài 3 — Biến thể: trả về vị trí

Trả về `(hàng, cột)` của `target`, hoặc `(-1, -1)` nếu không có. Giả sử các phần tử phân biệt.

In [20]:
def exercise_3_find_position(matrix: List[List[int]], target: int):
    # TODO
    pass


# assert exercise_3_find_position([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 16) == (1, 2)
# assert exercise_3_find_position([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 13) == (-1, -1)

## Bài 4 — Biến thể: Search a 2D Matrix II (LeetCode #240)

Mỗi hàng tăng dần, mỗi cột tăng dần (nhưng hàng sau **không nhất thiết** lớn hơn hàng trước). Tìm `target` trong `O(m + n)`.

```python
M = [[1, 4, 7, 11, 15],
     [2, 5, 8, 12, 19],
     [3, 6, 9, 16, 22],
     [10, 13, 14, 17, 24],
     [18, 21, 23, 26, 30]]
search_ii(M, 5)  -> True
search_ii(M, 20) -> False
```

In [21]:
def exercise_4_search_ii(matrix: List[List[int]], target: int) -> bool:
    # TODO
    pass


# M = [[1, 4, 7, 11, 15], [2, 5, 8, 12, 19], [3, 6, 9, 16, 22], [10, 13, 14, 17, 24], [18, 21, 23, 26, 30]]
# assert exercise_4_search_ii(M, 5) is True
# assert exercise_4_search_ii(M, 20) is False

## Bài 5 — Mini challenge: số phần tử nhỏ hơn hoặc bằng `x`

Với ma trận như đề #74 (có thể có phần tử trùng), đếm số phần tử `<= x` trong `O(log(m·n))`.

```python
count_le([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 11) -> 6
count_le([[1, 1], [1, 2]], 1) -> 3
```

Gợi ý nhanh: binary search tìm vị trí phẳng **đầu tiên** có giá trị `> x`.

In [22]:
def exercise_5_count_le(matrix: List[List[int]], x: int) -> int:
    # TODO
    pass


# assert exercise_5_count_le([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 11) == 6
# assert exercise_5_count_le([[1, 1], [1, 2]], 1) == 3
# assert exercise_5_count_le([[5]], 4) == 0

# 13. Gợi ý

**Bài 1**
- Mức 1: bước đầu `lo=0, hi=11, mid=5 -> (1,1) = 11`.

**Bài 3**
- Mức 1: như bản chính, trả `divmod(mid, n)` khi tìm thấy.

**Bài 4**
- Mức 1: bắt đầu ở góc **trên phải** (hoặc dưới trái). Mỗi bước loại được một hàng hoặc một cột.

**Bài 5**
- Mức 1: `lo, hi = 0, m*n` (nửa mở). Nếu `a(mid) <= x` thì `lo = mid + 1`, ngược lại `hi = mid`.
- Mức 2: kết thúc `lo == hi` là số phần tử `<= x`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [23]:
_ = trace_search(matrix, 30)

target = 30
  lo= 0 hi=11 mid= 5 -> (1,1) = 11  < target -> lo = 6
  lo= 6 hi=11 mid= 8 -> (2,0) = 23  < target -> lo = 9
  lo= 9 hi=11 mid=10 -> (2,2) = 34  > target -> hi = 9
  lo= 9 hi= 9 mid= 9 -> (2,1) = 30  == target -> True


## Đáp án Bài 2

In [24]:
def answer_2_search_matrix(matrix: List[List[int]], target: int) -> bool:
    m, n = len(matrix), len(matrix[0])
    lo, hi = 0, m * n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        r, c = divmod(mid, n)
        if matrix[r][c] == target:
            return True
        if matrix[r][c] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

assert answer_2_search_matrix([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 3) is True
assert answer_2_search_matrix([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 13) is False
assert answer_2_search_matrix([[1]], 1) is True
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [25]:
def answer_3_find_position(matrix: List[List[int]], target: int):
    m, n = len(matrix), len(matrix[0])
    lo, hi = 0, m * n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        r, c = divmod(mid, n)
        if matrix[r][c] == target:
            return (r, c)
        if matrix[r][c] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return (-1, -1)

assert answer_3_find_position([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 16) == (1, 2)
assert answer_3_find_position([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 13) == (-1, -1)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [26]:
def answer_4_search_ii(matrix: List[List[int]], target: int) -> bool:
    return search_staircase(matrix, target)

M = [[1, 4, 7, 11, 15], [2, 5, 8, 12, 19], [3, 6, 9, 16, 22], [10, 13, 14, 17, 24], [18, 21, 23, 26, 30]]
assert answer_4_search_ii(M, 5) is True
assert answer_4_search_ii(M, 20) is False
assert all(answer_4_search_ii(M, t) == search_linear(M, t) for t in range(0, 32))
print("Bài 4 PASS (giống search_staircase ở mục 7.2)")
print("Trải phẳng ma trận này có sắp xếp không?", [x for row in M for x in row] == sorted(x for row in M for x in row))

Bài 4 PASS (giống search_staircase ở mục 7.2)
Trải phẳng ma trận này có sắp xếp không? False


Dòng cuối cho thấy vì sao cách "trải phẳng" **không** dùng được cho #240: mảng phẳng không sắp xếp.

## Đáp án Bài 5

In [27]:
def answer_5_count_le(matrix: List[List[int]], x: int) -> int:
    m, n = len(matrix), len(matrix[0])
    lo, hi = 0, m * n
    while lo < hi:
        mid = (lo + hi) // 2
        if matrix[mid // n][mid % n] <= x:
            lo = mid + 1
        else:
            hi = mid
    return lo

assert answer_5_count_le([[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]], 11) == 6
assert answer_5_count_le([[1, 1], [1, 2]], 1) == 3
assert answer_5_count_le([[5]], 4) == 0
for _ in range(2000):
    m, n = random.randint(1, 5), random.randint(1, 5)
    values = sorted(random.randint(0, 10) for _ in range(m * n))
    mat = [values[i * n:(i + 1) * n] for i in range(m)]
    x = random.randint(-1, 11)
    assert answer_5_count_le(mat, x) == sum(v <= x for v in values)
print("Bài 5 PASS (2000 test có phần tử trùng)")

Bài 5 PASS (2000 test có phần tử trùng)


# 15. Tổng kết

> Ma trận #74 là một mảng sắp xếp bị cắt thành hàng. Binary search trên chỉ số `0 .. m·n − 1`, truy cập `matrix[k // n][k % n]`. `O(log(m·n))` thời gian, `O(1)` bộ nhớ.

---

## Checklist trước khi nộp

- [ ] Chia cho **`n` (số cột)** khi đổi chỉ số?
- [ ] `while lo <= hi` đi với `lo = mid + 1`, `hi = mid - 1`?
- [ ] Không tạo mảng phẳng mới?
- [ ] Biết dùng cách cầu thang khi đề là #240?